# SAM 3 on S2DR4 1 m imagery — Biak burn scars

Moves the SAM work off the laptop, which lost power three times under GPU load (Kernel-Power 41, twice mid-run).

**What Linux buys us.** On Windows only text prompts run: the Meta backend needs `triton`, which has no Windows build, and point and box prompts are Meta-only. Here both work, so the 10 m pilot's point-prompt comparison can be repeated at 1 m.

**What is already known** (laptop, 2 x 2 km crop of E0313 at 1 m):

| prompt | true colour | SWIR false colour |
|---|---|---|
| tree | 50 masks | 0 |
| vegetation | 95 | 0 |
| building | 29 | 17 |
| road | 7 | 2 |
| **burned area** | **0** | **0** |

at confidence 0.15 and 0.5. So the pipeline and the imagery are fine, and the model simply has no grounding for a burn scar. SAM 2.1 at 10 m failed differently: point prompts returned one blob covering all land, and automatic masking found 2-4 objects per chip.

This notebook settles the rest: many wordings, point prompts at the VIIRS detections, box prompts, all scored against the Task 18 dNBR+ polygons.

**Runtime → Change runtime type → GPU** (an L4 or A100 on Colab Pro is plenty).

## 1. Install

In [ ]:
!pip -q install "segment-geospatial[samgeo3]" rasterio shapely pyproj pandas pyarrow
# LangSAM (step 6c) drives GroundingDINO, whose BERT warper was written for the transformers 4.x API.
# The samgeo3 extra above pulls transformers 5, which removed that API, so pin 4.x back after it.
!pip -q install "transformers<5"
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

## 2. Hugging Face sign-in

SAM 3 is gated. Paste the same read token as in `.env` when prompted — it is not stored in the notebook.

In [ ]:
from getpass import getpass
from huggingface_hub import login, HfApi

login(token=getpass('HF token: '))
print('account:', HfApi().whoami().get('name'))

## 3. Inputs

The events, the detection store and the dNBR+ polygons come from the public repo, so nothing has to be uploaded by hand. The 1 m imagery is the S2DR4 output: copy the extracted `s2dr4_biak_2026-08-28` folder to Drive first (about 700 MB), then point `S2DR4` at it.

In [ ]:
!git clone -q --depth 1 https://github.com/daniel-sb/biak-hotspot /content/biak

from google.colab import drive
drive.mount('/content/drive')

# adjust if you put the folder elsewhere in Drive
S2DR4 = '/content/drive/MyDrive/s2dr4_biak_2026-08-28/output/ID/T53MPU'

import glob
print('\n'.join(sorted(glob.glob(f'{S2DR4}/*/*_MS.tif'))) or 'NOT FOUND - fix the S2DR4 path')

## 4. Crops, footprints and the dNBR+ reference

One 2 x 2 km crop per event, centred on its detections, in two renderings: true colour (`TCI`) and SWIR false colour (B12/B8A/B4 from `MS`, 2-98% stretch). Burn scars read best in SWIR; SAM was trained on ordinary photographs, which is what TCI resembles. Testing both is the point.

In [ ]:
import json, sys, numpy as np, pandas as pd, rasterio
from pathlib import Path
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import shape, mapping
from shapely.ops import transform as shp_transform
from pyproj import Transformer

REPO = Path('/content/biak')
sys.path.insert(0, str(REPO / 'src'))
import events as ev, burned_area_gee as ba

TILE = {'E0313': 'T53MPU-0c59f172e', 'E0301': 'T53MPU-4e09bbfee', 'E0368': 'T53MPU-c501689c0'}
SIZE = 2048            # metres, at 1 m per pixel
OUT = Path('/content/work'); OUT.mkdir(exist_ok=True)
to_utm = Transformer.from_crs('EPSG:4326', 'EPSG:32753', always_xy=True).transform

doc = json.loads((REPO / 'data/processed/events.json').read_text())
mem = ev.membership(doc)
store = pd.read_parquet(REPO / 'data/processed/detections.parquet')
polys = json.loads((REPO / 'data/processed/burned_areas.geojson').read_text())['features']

chips = {}
for eid, tile in TILE.items():
    ids = [d for d, e in mem.items() if e == eid]
    pts = store[store.detection_id.isin(ids)][['latitude', 'longitude']]
    xs, ys = zip(*[to_utm(lon, lat) for lat, lon in pts.values])
    cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2

    ms = f'{S2DR4}/{tile}/S2L3Ax10_{tile}-20260828_MS.tif'
    with rasterio.open(ms) as r:
        row, col = r.index(cx, cy)
        win = Window(col - SIZE // 2, row - SIZE // 2, SIZE, SIZE)
        prof = r.profile.copy()
        prof.update(height=SIZE, width=SIZE, count=3, dtype='uint8', nodata=None,
                    transform=r.window_transform(win))
        swir = r.read(indexes=[10, 8, 3], window=win).astype('float32')   # B12, B8A, B4
    out = np.zeros(swir.shape, 'uint8')
    for i, band in enumerate(swir):
        lo, hi = np.percentile(band, [2, 98])
        out[i] = np.clip((band - lo) / max(hi - lo, 1e-6) * 255, 0, 255)
    swir_p = OUT / f'{eid}_swir.tif'
    with rasterio.open(swir_p, 'w', **prof) as w:
        w.write(out)

    tci = f'{S2DR4}/{tile}/S2L3Ax10_{tile}-20260828_TCI.tif'
    with rasterio.open(tci) as r:
        a = r.read(window=win)[:3]
    tci_p = OUT / f'{eid}_tci.tif'
    with rasterio.open(tci_p, 'w', **prof) as w:
        w.write(a.astype('uint8'))

    fp_geo, _ = ba.footprint(pts.latitude.tolist(), pts.longitude.tolist(), 375.0)
    shp = lambda g: rasterize([(mapping(g), 1)], out_shape=(SIZE, SIZE),
                              transform=prof['transform'], fill=0, dtype='uint8').astype(bool)
    fp = shp(shp_transform(to_utm, fp_geo))
    ref = np.zeros((SIZE, SIZE), bool)
    for f in polys:
        if f['properties']['event_id'] == eid:
            ref |= shp(shp_transform(to_utm, shape(f['geometry'])))
    px_ha = abs(prof['transform'].a * prof['transform'].e) / 1e4
    chips[eid] = {'swir': swir_p, 'tci': tci_p, 'fp': fp, 'ref': ref, 'px_ha': px_ha,
                  'points': pts, 'profile': prof}
    print(f'{eid}: {len(pts):3d} detections | footprint in crop {fp.sum() * px_ha:6.1f} ha | '
          f'dNBR+ in crop {ref.sum() * px_ha:6.1f} ha')

## 5. Does the wording matter?

Eleven ways of naming a burn scar, plus `tree` and `building` as controls that prove the model is working on this imagery. The laptop run got through four wordings before it died; this finishes the list.

In [ ]:
from samgeo import SamGeo3

WORDS = ['burned area', 'burn scar', 'fire scar', 'burnt land', 'charred ground', 'ash',
         'black soil', 'brown field', 'cleared land', 'dry grass', 'farm field']
CONTROLS = ['tree', 'building']

sam = SamGeo3(backend='meta', load_from_HF=True, confidence_threshold=0.15)
counts = {}
for eid in TILE:
    for render in ('tci', 'swir'):
        sam.set_image(str(chips[eid][render]))
        for w in WORDS + CONTROLS:
            try:
                sam.generate_masks(prompt=w)
                n = len(getattr(sam, 'masks', []) or [])
            except Exception:
                n = 0
            counts[f'{eid}|{render}|{w}'] = n
            print(f'{eid} {render:4s} {w:15s} -> {n:3d} masks', flush=True)

json.dump(counts, open(OUT / 'sam3_word_counts.json', 'w'), indent=1, sort_keys=True)

## 6. Point prompts at the VIIRS detections

The comparison the laptop could not run. Each detection is one prompt; the union of the masks is scored against the dNBR+ polygons inside the event footprint. The prompts come from VIIRS, not from dNBR+, so the comparison is not circular.

At 10 m with SAM 2.1 this returned a blob covering all land: 924 ha of a 924 ha footprint plus 1,914 ha outside it.

In [ ]:
def score(mask, eid):
    c = chips[eid]
    s, r = mask & c['fp'], c['ref'] & c['fp']
    inter, union = int((s & r).sum()), int((s | r).sum())
    return {'sam_ha_in_footprint': round(s.sum() * c['px_ha'], 1),
            'dnbr_ha_in_footprint': round(r.sum() * c['px_ha'], 1),
            'iou': round(inter / union, 3) if union else None,
            'recall_of_dnbr': round(inter / r.sum(), 3) if r.sum() else None,
            'sam_ha_outside_footprint': round(int((mask & ~c['fp']).sum()) * c['px_ha'], 1)}

sam_pt = SamGeo3(backend='meta', load_from_HF=True, enable_inst_interactivity=True)
results = {}
for eid in TILE:
    c = chips[eid]
    for render in ('tci', 'swir'):
        sam_pt.set_image(str(c[render]))
        path = OUT / f'{eid}_{render}_points.tif'
        try:
            sam_pt.generate_masks_by_points_patch(
                point_coords_batch=[[float(lon), float(lat)] for lat, lon in
                                    c['points'][['latitude', 'longitude']].values],
                point_crs='EPSG:4326', output=str(path), dtype='uint16')
            with rasterio.open(path) as r:
                results[f'{eid}|{render}|points'] = score(r.read(1) > 0, eid)
        except Exception as e:
            results[f'{eid}|{render}|points'] = {'error': f'{type(e).__name__}: {str(e)[:80]}'}
        print(eid, render, results[f'{eid}|{render}|points'], flush=True)

json.dump(results, open(OUT / 'sam3_point_scores.json', 'w'), indent=1, sort_keys=True)

## 6b. Box prompts

Boxes come from the VIIRS detections, never from dNBR+, so the comparison stays honest: one 750 m box per detection (the 375 m footprint radius, which is one VIIRS pixel), plus one box around all of an event's detections.

The expectation is another whole-land blob. The point-prompt masks covered 100% of every footprint, which is why their IoU equals dNBR+ divided by footprint exactly — 0.732, 0.579 and 0.307. A mask that simply claims the whole footprint burned scores the same. If box prompts break that pattern, the numbers below will differ from those three.

In [ ]:
import numpy as np, rasterio

M_PER_DEG = 111_320.0
R = 375.0                                   # one VIIRS pixel, as in Task 18

sam_box = SamGeo3(backend='meta', load_from_HF=True, enable_inst_interactivity=True)
box_results = {}
for eid in TILE:
    c = chips[eid]
    lat = c['points'].latitude.values
    lon = c['points'].longitude.values
    dlat = R / M_PER_DEG
    dlon = R / (M_PER_DEG * np.cos(np.radians(lat)))
    per_det = [[float(lo - dx), float(la - dlat), float(lo + dx), float(la + dlat)]
               for lo, la, dx in zip(lon, lat, dlon)]
    whole = [[float(lon.min() - dlon.mean()), float(lat.min() - dlat),
              float(lon.max() + dlon.mean()), float(lat.max() + dlat)]]

    for render in ('tci', 'swir'):
        sam_box.set_image(str(c[render]))
        for label, boxes in (('per_detection', per_det), ('one_box', whole)):
            key = f'{eid}|{render}|box_{label}'
            path = OUT / f'{eid}_{render}_box_{label}.tif'
            try:
                sam_box.generate_masks_by_boxes(boxes, box_crs='EPSG:4326')
                sam_box.save_masks(output=str(path), unique=True)
                with rasterio.open(path) as r:
                    box_results[key] = score(r.read(1) > 0, eid)
            except Exception as e:
                box_results[key] = {'error': f'{type(e).__name__}: {str(e)[:80]}'}
            print(key, box_results[key], flush=True)

json.dump(box_results, open(OUT / 'sam3_box_scores.json', 'w'), indent=1, sort_keys=True)

# Does the mask simply swallow the footprint, as the point prompts did?
for k, v in box_results.items():
    if 'error' not in v:
        eid = k.split('|')[0]
        fp_ha = round(chips[eid]['fp'].sum() * chips[eid]['px_ha'], 1)
        print(f"{k:34s} sam_in_fp {v['sam_ha_in_footprint']:7.1f} of footprint {fp_ha:7.1f} ha"
              f"  -> {'covers the whole footprint' if v['sam_ha_in_footprint'] >= fp_ha - 0.5 else 'TRACES SOMETHING'}")


## 6c. LangSAM — the other text route in samgeo

SAM 3's own text head found nothing for eleven burn-scar wordings, while finding 25 trees and 40 buildings in the same imagery. `samgeo.text_sam.LangSAM` works differently: **Grounding DINO** proposes boxes from a phrase, and **SAM 1** segments inside them. The detector is a separate model with its own vocabulary, so the phrases it grounds are not the phrases SAM 3 grounds.

Two thresholds matter here. `box_threshold` is how confident the detector must be to propose a box, `text_threshold` how well the phrase must match. Both are set low (0.15 / 0.15) on purpose: a strict setting would hide the difference between "the concept is absent" and "the model is being cautious".

In [ ]:
!pip -q install "segment-geospatial[text]" "transformers<5" 2>/dev/null || pip -q install groundingdino-py segment-anything "transformers<5"

import transformers
if int(transformers.__version__.split('.')[0]) >= 5:
    raise RuntimeError('transformers 5 is loaded in this kernel; GroundingDINO needs the 4.x API. '
                       'Restart the session (Runtime -> Restart session) and re-run from step 1.')

from samgeo.text_sam import LangSAM
from PIL import Image
import numpy as np, rasterio, json

PHRASES = ['burned area', 'burn scar', 'burnt land', 'dark patch of land',
           'bare soil', 'cleared land', 'tree', 'building']
BOX_T, TEXT_T = 0.15, 0.15

lang = LangSAM(model_type='vit_h')
lang_results = {}
for eid in TILE:
    c = chips[eid]
    for render in ('tci', 'swir'):
        src = str(c[render])
        for phrase in PHRASES:
            key = f'{eid}|{render}|lang:{phrase}'
            out_tif = OUT / f"{eid}_{render}_lang_{phrase.replace(' ', '_')}.tif"
            try:
                lang.predict(src, phrase, BOX_T, TEXT_T, output=str(out_tif))
                n_boxes = 0 if getattr(lang, 'boxes', None) is None else len(lang.boxes)
                if out_tif.exists():
                    with rasterio.open(out_tif) as r:
                        m = r.read(1) > 0
                    lang_results[key] = {'n_boxes': n_boxes, **score(m, eid)}
                else:
                    lang_results[key] = {'n_boxes': n_boxes, 'note': 'no mask written'}
            except Exception as e:
                lang_results[key] = {'error': f'{type(e).__name__}: {str(e)[:80]}'}
            print(key, lang_results[key], flush=True)

json.dump(lang_results, open(OUT / 'sam3_lang_scores.json', 'w'), indent=1, sort_keys=True)

# the bar to beat: claiming the whole footprint burned
print()
for eid in TILE:
    c = chips[eid]
    trivial = (c['ref'] & c['fp']).sum() / max(c['fp'].sum(), 1)
    best = max((v.get('iou') or 0 for k, v in lang_results.items()
                if k.startswith(eid) and 'error' not in v), default=0)
    print(f"{eid}: trivial claim {trivial:.3f} | best LangSAM {best:.3f} -> "
          f"{'beats it' if best > trivial else 'does not beat it'}")


## 7. Overlays

Red is the model mask, blue the dNBR+ polygons. The last column is each event's best LangSAM mask (highest IoU; phrase and render are in the title). A picture settles what the numbers only hint at: whether the model traced the scar or simply segmented the land.

In [ ]:
ce

## 8. Download the results

Small files only: the four score files (words, points, boxes, LangSAM) and the overlay figure.

In [ ]:
from google.colab import files
!cd /content/work && zip -q sam3_results.zip sam3_word_counts.json sam3_point_scores.json sam3_box_scores.json sam3_lang_scores.json sam3_overlays.png
files.download('/content/work/sam3_results.zip')